In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob

# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px
import func_graff as graff

**Insumos**

In [84]:
periodo_analisis = "202607"
ruta = r"bd\bd_productividad.xlsx"
ruta_velocdidad = r"bd\bd_velocidad.xlsx"

In [85]:
#TIPO DE VENTA: LAST MILE + ECOMMERCE + POS

df_prod = pd.read_excel(ruta)

In [86]:
df_locales = pd.read_excel(r"bd\dim_locales.xlsx")
df_locales.rename(columns={"Tienda": "Tienda2"}, inplace=True)
df_locales.rename(columns={"_Tienda": "Tienda"}, inplace=True)

In [87]:
df_prod["GOR"].unique()

array([nan, 'Joe H', 'Vik E', 'Rodolfo O', 'José A', 'Daniel R',
       'Johanna V', 'Melany A', 'Fátima L', '--', 'María R'], dtype=object)

In [88]:

df_prod1 = df_prod[df_prod["GOS"].isin(['Fátima L', 'Sofía V', 'Sebastián S', 'Hernán P', 'María R'])][['Tienda', 'Periodo', 'GOS', 'GOR', 'Prod', 'VtaNeta', 'JEq',
       'ProdMeta', 'VtaNetaMeta', 'JEqMeta', 'ProdAA', 'VtaNetaAA', 'JEqAA']]

# df_prod1 = df_prod1.merge(df_locales[["Tienda", "GOR"]], on='Tienda', how='left')
df_prod1 = df_prod1[(df_prod1["GOR"]=="Melany A")&(df_prod1["Tienda"]!="P075 Trujillo - PVH")]

In [89]:
df_prod1["Tienda"].unique()

array(['P102 Chiclayo - PVH', 'P123 El Chacarero - PVH',
       'P130 Chimbote - PVH', 'P143 Nvo Chimbote - PVH',
       'P145 Piura - PVH', 'P177 Talara - PVH', 'P192 Sullana - PVH',
       'P226 Paita - PVH', 'P262 Talara Municipalidad - PVH',
       'P724 Tumbes - PVH', 'P769 Chiclayo Aventura - PVH'], dtype=object)

In [90]:
df_prod1["Tienda"].nunique()

11

**Productividad**

In [91]:
df_prod1["Periodo"] = df_prod1["Periodo"].astype(int).astype(str)
df_prod2 = df_prod1[df_prod1["Periodo"]==periodo_analisis]
venta = df_prod2["VtaNeta"].sum()
jeq = df_prod2["JEq"].sum()
venta_meta = df_prod2["VtaNetaMeta"].sum()
jeq_meta = df_prod2["JEqMeta"].sum()
venta_aa = df_prod2["VtaNetaAA"].sum()
jeq_aa = df_prod2["JEqAA"].sum()


prod = venta/jeq
prod_meta = venta_meta/jeq_meta
prod_aa = venta_aa/jeq_aa

var_meta = prod/prod_meta
var_aa = prod/prod_aa

In [92]:
df_prod1["año"] = df_prod1["Periodo"].str[0:4]
df_prod1["mes"] = df_prod1["Periodo"].str[-2:]

dffx = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"VtaNeta":"sum", "JEq":"sum","VtaNetaMeta":"sum","JEqMeta":"sum"}).reset_index()
dffy = df_prod1[(df_prod1["año"]=="2025")].groupby(["mes"]).agg(VtaAA=("VtaNeta","sum"), JEqAA = ("JEq","sum")).reset_index()
dffz = dffy.merge(dffx, on="mes", how="left")
dffz["Prod"] = dffz["VtaNeta"] / dffz["JEq"]
dffz["ProdMeta"] = dffz["VtaNetaMeta"] / dffz["JEqMeta"]
dffz["ProdAA"] = dffz["VtaAA"] / dffz["JEqAA"]
dffz["Cumplimiento_Meta"] = dffz["Prod"]/dffz["ProdMeta"]-1
dffz

,mes,VtaAA,JEqAA,VtaNeta,JEq,VtaNetaMeta,JEqMeta,Prod,ProdMeta,ProdAA,Cumplimiento_Meta
0,01,59391442.51,701.341714,6.510076e+07,691.222668,6.584307e+07,712.806579,94182.036560,92371.577114,84682.603856,0.019600
1,02,59643536.44,723.586406,6.936358e+07,704.604635,6.457419e+07,712.806579,98443.260622,90591.457295,82427.662992,0.086673
2,03,80567392.26,760.302460,8.354588e+07,747.838367,8.416032e+07,712.806579,111716.496163,118068.938300,105967.554405,-0.053803
3,04,63739174.84,766.078299,6.896095e+07,698.907222,6.749030e+07,712.806579,98669.680620,94682.484878,83201.906327,0.042111
4,05,64322764.49,772.797883,6.931286e+07,703.081599,6.941382e+07,712.806579,98584.374976,97381.004677,83233.618906,0.012357
5,06,57062302.14,766.935896,6.130459e+07,688.568125,6.214793e+07,712.806579,89031.986748,87187.648357,74402.961773,0.021154
6,07,63762571.67,753.895013,6.219103e+07,695.120276,6.732855e+07,712.806579,89468.006109,94455.566213,84577.521450,-0.052803
7,08,60058444.00,712.404489,NaN,NaN,NaN,NaN,NaN,NaN,84303.853929,NaN
8,09,55034222.34,697.778861,NaN,NaN,NaN,NaN,NaN,NaN,78870.578355,NaN
9,10,57649320.29,689.943663,NaN,NaN,NaN,NaN,NaN,NaN,83556.561807,NaN


In [93]:
graff.KPI_Productividad_HTML(
    prod,
    "PRODUCTIVIDAD",
    prod_meta,
    prod_aa,
    var_meta,
    var_aa,
    r"plenario/kpi_productividad.html"
)


In [94]:
graff.Historico_Comparativo(
    df=dffz,
    anio1="ProdAA",
    anio2="Prod",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Productividad -  Histórico",
    archivo_html=r"plenario/Historico_Productividad.html",
    ymin=None,
    ymax=None,
    y2min=-0.4,
    y2max=0.25,
    col_mes="mes"
)

In [95]:
df_prod3 = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)]

df_gor_historico = df_prod3.groupby(["Tienda", "mes"]).agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gor_historico["Prod"] = df_gor_historico["VtaNeta"] / df_gor_historico["JEq"]
df_gor_historico["ProdMeta"] = df_gor_historico["VtaNetaMeta"] / df_gor_historico["JEqMeta"]
df_gor_historico["ProdAA"] = df_gor_historico["VtaNetaAA"] / df_gor_historico["JEqAA"]
df_gor_historico["Cump_Meta"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]-1
df_gor_historico["Cump_Meta2"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]
df_gor_historico["Cump_AA"] = df_gor_historico["Prod"]/df_gor_historico["ProdAA"]-1
df_gor_historico = df_gor_historico.sort_values(by="Cump_Meta", ascending=False)
df_gor_historico.head()

,Tienda,mes,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
73,P769 Chiclayo Aventura - PVH,04,6000819.73,48.304132,5.050431e+06,53.278950,4984716.14,57.339722,124229.946558,94792.233174,86933.036067,0.310550,1.310550,0.429030
8,P123 El Chacarero - PVH,02,9907585.09,105.651719,7.452820e+06,102.976121,7387252.51,79.913385,93775.900735,72374.255860,92440.740328,0.295708,1.295708,0.014443
29,P145 Piura - PVH,02,11615478.75,90.897917,1.030808e+07,102.976121,8959162.89,107.528437,127785.973276,100101.636341,83319.009355,0.276562,1.276562,0.533695
74,P769 Chiclayo Aventura - PVH,05,6281220.46,50.000464,5.365252e+06,53.278950,5002160.18,58.364005,125623.244146,100701.166094,85706.252471,0.247485,1.247485,0.465742
75,P769 Chiclayo Aventura - PVH,06,5307337.41,49.046083,4.724861e+06,53.278950,4358585.63,56.026687,108211.238274,88681.579230,77794.812160,0.220222,1.220222,0.390983


In [96]:
graff.Heatmap_GOR(
    df_gor_historico,
    gor="Tienda",
    mes="mes",
    mes_prior = "Jul",
    valor="Cump_Meta",
    archivo_html=r"plenario/Heatmap_GOR.html",
    titulo="Cumplimiento vs Meta")
# graff.HTML_a_PNG( r"plenario/Heatmap_GOR.html", r"plenario/02_Dashboard_historico_GOR.png")

Archivo generado: plenario/Heatmap_GOR.html


In [97]:
df_tienda = df_prod2.groupby("Tienda").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_tienda["Prod"] = df_tienda["VtaNeta"] / df_tienda["JEq"]
df_tienda["ProdMeta"] = df_tienda["VtaNetaMeta"] / df_tienda["JEqMeta"]
df_tienda["ProdAA"] = df_tienda["VtaNetaAA"] / df_tienda["JEqAA"]
df_tienda["Cump_Meta"] = df_tienda["Prod"]/df_tienda["ProdMeta"]-1
df_tienda["Cump_Meta2"] = df_tienda["Prod"]/df_tienda["ProdMeta"]
df_tienda["Cump_AA"] = df_tienda["Prod"]/df_tienda["ProdAA"]-1
df_tienda = df_tienda.sort_values(by="Cump_Meta", ascending=False)
df_tienda1 = df_tienda[
    np.isfinite(df_tienda["Prod"]) &
    np.isfinite(df_tienda["ProdMeta"]) &
    (df_tienda["Prod"] > 1) &
    (df_tienda["ProdMeta"] > 1)
].copy()

regex_limpieza = r"^[A-Z0-9 ]+\s(?=[A-Z][a-z])|\s*-\s*[A-Z]+$"

df_tienda1["Tienda"] = df_tienda1["Tienda"].str.replace(
    regex_limpieza, "", regex=True
)
df_tienda1_topx = df_tienda1.sort_values(by="Cump_Meta", ascending=False)
df_tienda1_top = df_tienda1_topx[~df_tienda1_topx["Tienda"].isin(["Primavera", "Asia"])]
df_tienda1_top = df_tienda1_top.head(6)

df_tienda1_bot = df_tienda1.sort_values(by="Cump_Meta", ascending=True)
df_tienda1_bot = df_tienda1_bot.head(6)

In [98]:
graff.Barras_GOR(
    df_tienda1_top,
    gor="Tienda",
    ancho="Cump_Meta2",
    cump_meta="Cump_Meta",
    cump_aa="Cump_AA",
    prod="Prod",
    titulo="Productividad por GOR",
    archivo_html=r"plenario/Barras_top_GOR.html"
)

Archivo generado: plenario/Barras_top_GOR.html


In [99]:
graff.Barras_GOR(
    df_tienda1_bot,
    gor="Tienda",
    ancho="Cump_Meta2",
    cump_meta="Cump_Meta",
    cump_aa="Cump_AA",
    prod="Prod",
    titulo="Productividad por GOR",
    archivo_html=r"plenario/Barras_bot_GOR.html"
)

Archivo generado: plenario/Barras_bot_GOR.html


**Velocidad**

In [100]:
df_veloc = pd.read_excel(ruta_velocdidad)

df_veloc["GOS"] = ( df_veloc["regional"].str.split().str[0]  + " " + df_veloc["regional"].str.split().str[1].str[0])
df_veloc["GOR"] = (df_veloc["supervisor"].str.title().str.split().str[0] + " " + df_veloc["supervisor"].str.title().str.split().str[1].str[0])

meses_dict = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12"
}



In [101]:
df_veloc = df_veloc.dropna(subset=["Date - Año"])

In [102]:
df_veloc["mes"] = df_veloc["Date - Mes"].str.lower().map(meses_dict)
df_veloc["año"] = df_veloc["Date - Año"].astype(int).astype(str)
df_veloc["Periodo"] = df_veloc["año"].astype(str) + df_veloc["mes"]
df_veloc["Tienda"] = df_veloc["nombre"].str.split(" - ").str[0]

df_aa = ( df_veloc[["Tienda", "mes", "año", "venta_unidad", "tiempo_escaneo"]].rename(columns={"venta_unidad": "venta_unidad_aa", "tiempo_escaneo": "tiempo_escaneo_aa"}))
df_aa["año"] = df_aa["año"].astype(int)
df_aa["año"] = df_aa["año"] + 1
df_aa["año"] = df_aa["año"].astype(str)

df_veloc1 = df_veloc.merge( df_aa, on=["Tienda", "mes", "año"], how="left")

In [103]:
df_veloc["supervisor"].unique()

array(['Fátima León', 'DANIEL RODRIGUEZ', 'María Rojas', 'JOHANNA VILCA',
       'RODOLFO OLIVRY', 'VIK ENCISO', 'JOE HUAMANCONDOR',
       'MELANY ALVARADO', 'PEPE ARAMBURU'], dtype=object)

In [104]:
df_veloc1 = df_veloc1[df_veloc1["supervisor"] == "MELANY ALVARADO"]
df_veloc1.head()

,regional,supervisor,Date - Año,Date - Mes,nombre,Unid x Minuto,Meta Promedio,venta_unidad,tiempo_escaneo,Objetivo,GOS,GOR,mes,año,Periodo,Tienda,venta_unidad_aa,tiempo_escaneo_aa
2136,Sofía Villanueva,MELANY ALVARADO,2025.0,enero,Chiclayo - PVH,13.268870,14.0,812041.776,3671941.0,14.0,Sofía V,Melany A,01,2025,202501,Chiclayo,NaN,NaN
2137,Sofía Villanueva,MELANY ALVARADO,2025.0,enero,Chiclayo Aventura - PVH,14.268676,14.0,491047.220,2064861.0,14.0,Sofía V,Melany A,01,2025,202501,Chiclayo Aventura,NaN,NaN
2138,Sofía Villanueva,MELANY ALVARADO,2025.0,enero,Chimbote - PVH,14.553288,15.0,614479.835,2533365.0,15.0,Sofía V,Melany A,01,2025,202501,Chimbote,NaN,NaN
2139,Sofía Villanueva,MELANY ALVARADO,2025.0,enero,El Chacarero - PVH,15.618287,14.0,646067.350,2481965.0,14.0,Sofía V,Melany A,01,2025,202501,El Chacarero,NaN,NaN
2140,Sofía Villanueva,MELANY ALVARADO,2025.0,enero,Multiplaza - PVS,NaN,13.0,NaN,NaN,13.0,Sofía V,Melany A,01,2025,202501,Multiplaza,NaN,NaN


In [105]:
df_veloc1["nombre"].unique()

array(['Chiclayo - PVH', 'Chiclayo Aventura - PVH', 'Chimbote - PVH',
       'El Chacarero - PVH', 'Multiplaza - PVS', 'Nvo Chimbote - PVH',
       'Paita - PVH', 'Piura - PVH', 'Sullana - PVH', 'Talara - PVH',
       'Talara Municipalidad - PVH', 'Trujillo - PVH', 'Tumbes - PVH'],
      dtype=object)

In [106]:
df_veloc2 = df_veloc1[(df_veloc1["Periodo"]==periodo_analisis)&(df_veloc1["nombre"]!="Trujillo - PVH")]
df_veloc2

venta_unid = df_veloc2["venta_unidad"].sum()
tiempo_scan = df_veloc2["tiempo_escaneo"].sum()
venta_unid_aa = df_veloc2["venta_unidad_aa"].sum()
tiempo_scan_aa = df_veloc2["tiempo_escaneo_aa"].sum()


velocidad = venta_unid/tiempo_scan*60
velocidad_aa = venta_unid_aa/tiempo_scan_aa*60
velocidad_meta = df_veloc2["Objetivo"].mean()

var_veloc_meta = velocidad/velocidad_meta
var_veloc_aa = velocidad/velocidad_aa


In [107]:
df_veloc1x = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"venta_unidad":"sum", "tiempo_escaneo":"sum","Objetivo":"mean"}).reset_index()
df_veloc1y = df_veloc1[(df_veloc1["año"]=="2025")].groupby(["mes"]).agg(venta_unidad_AA=("venta_unidad","sum"), tiempo_escaneo_AA = ("tiempo_escaneo","sum")).reset_index()
df_veloc1z = df_veloc1y.merge(df_veloc1x, on="mes", how="left")
df_veloc1z["Velocidad"] = df_veloc1z["venta_unidad"] / df_veloc1z["tiempo_escaneo"]*60
df_veloc1z["VelocidadMeta"] = df_veloc1z["Objetivo"]
df_veloc1z["VelocidadAA"] = df_veloc1z["venta_unidad_AA"] / df_veloc1z["tiempo_escaneo_AA"]*60
df_veloc1z["Cumplimiento_Meta"] = df_veloc1z["Velocidad"]/df_veloc1z["VelocidadMeta"]-1
df_veloc1z

,mes,venta_unidad_AA,tiempo_escaneo_AA,venta_unidad,tiempo_escaneo,Objetivo,Velocidad,VelocidadMeta,VelocidadAA,Cumplimiento_Meta
0,01,6508682.568,27724780.0,6098462.13,27074620.0,15.076923,13.514787,15.076923,14.085629,-0.103611
1,02,6305142.238,26437120.0,6134032.55,25878704.0,15.076923,14.221808,15.076923,14.309748,-0.056717
2,03,8189313.037,33302986.0,7462988.65,31044516.0,15.076923,14.423782,15.076923,14.754196,-0.043321
3,04,6719038.382,27836201.0,5858142.50,24548016.0,15.076923,14.318410,15.076923,14.482662,-0.050310
4,05,6905239.810,28436603.0,6344790.36,26701341.0,15.076923,14.257240,15.076923,14.569757,-0.054367
5,06,6352078.920,27286246.0,5554843.51,23095473.0,15.076923,14.430993,15.076923,13.967650,-0.042842
6,07,6519049.710,27270022.0,2345587.21,9889003.0,15.076923,14.231489,15.076923,14.343332,-0.056075
7,08,6321077.600,26942542.0,NaN,NaN,NaN,NaN,NaN,14.076796,NaN
8,09,6019892.220,23679495.0,NaN,NaN,NaN,NaN,NaN,15.253431,NaN
9,10,6162051.320,24460515.0,NaN,NaN,NaN,NaN,NaN,15.115098,NaN


In [108]:
graff.KPI_Productividad_HTML(
    velocidad,
    "VELOCIDAD",
    velocidad_meta,
    velocidad_aa,
    var_veloc_meta,
    var_veloc_aa,
    r"plenario/kpi_velocidad.html"
)



In [109]:
graff.Historico_Comparativo(
    df=df_veloc1z,
    anio1="VelocidadAA",
    anio2="Velocidad",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Velocidad -  Histórico",
    archivo_html=r"plenario/Historico_Velocidad.html",
    ymin=None,
    ymax=None,
    y2min=-0.8,
    y2max=0.10,
    col_mes="mes"
)

In [110]:
df_veloc3 = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)]

df_gor_veloc_hist = df_veloc3.groupby(["nombre", "mes"]).agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_gor_veloc_hist["Velocidad"] = df_gor_veloc_hist["venta_unidad"] / df_gor_veloc_hist["tiempo_escaneo"]*60
df_gor_veloc_hist["VelocidadMeta"] = df_gor_veloc_hist["Objetivo"]
df_gor_veloc_hist["VelocidadAA"] = df_gor_veloc_hist["venta_unidad_aa"] / df_gor_veloc_hist["tiempo_escaneo_aa"]*60
df_gor_veloc_hist["Cump_Meta"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]-1
df_gor_veloc_hist["Cump_Meta2"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]
df_gor_veloc_hist["Cump_AA"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadAA"]-1
df_gor_veloc_hist = df_gor_veloc_hist.sort_values(by="Cump_Meta", ascending=False)
df_gor_veloc_hist.head()

,nombre,mes,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
44,Paita - PVH,03,688490.42,2476124.0,14.0,671698.165,2875856.0,16.683100,14.0,14.013876,0.191650,1.191650,0.190470
43,Paita - PVH,02,540652.29,1971851.0,14.0,526632.080,2374397.0,16.451110,14.0,13.307768,0.175079,1.175079,0.236204
45,Paita - PVH,04,555275.55,2036630.0,14.0,567033.868,2326142.0,16.358658,14.0,14.625948,0.168476,1.168476,0.118468
46,Paita - PVH,05,599588.40,2204519.0,14.0,583209.820,2337303.0,16.318890,14.0,14.971353,0.165635,1.165635,0.090008
47,Paita - PVH,06,505276.49,1924856.0,14.0,517086.460,2257238.0,15.750056,14.0,13.744757,0.125004,1.125004,0.145896


In [111]:
graff.Heatmap_GOR(
    df_gor_veloc_hist,
    gor="nombre",
    mes="mes",
    valor="Cump_Meta",
    archivo_html=r"plenario/Heatmap_Velocidad_GOR.html",
    titulo="Cumplimiento vs Meta")


Archivo generado: plenario/Heatmap_Velocidad_GOR.html


In [112]:
df_tienda1_top

,Tienda,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
10,Chiclayo Aventura,4.951635e+06,49.346707,4.927617e+06,53.278950,4907998.91,57.160363,100343.775555,92487.127748,85863.676518,0.084949,1.084949,0.168641
8,Talara Municipalidad,5.161840e+06,52.529120,4.919340e+06,53.343390,4789951.84,55.541190,98266.250286,92220.227540,86241.434181,0.065561,1.065561,0.139432
4,Piura,9.777505e+06,102.227332,9.767261e+06,102.976121,9455799.67,108.033663,95644.721670,94849.769875,87526.419446,0.008381,1.008381,0.092753
0,Chiclayo,8.410205e+06,79.777749,9.244938e+06,87.753564,8758050.83,92.139570,105420.440370,105351.143462,95052.004695,0.000658,1.000658,0.109082
2,Chimbote,5.088824e+06,58.098212,5.839903e+06,66.569024,5702562.44,62.560618,87590.032574,87727.036610,91152.590828,-0.001562,0.998438,-0.039083
1,El Chacarero,9.239349e+06,103.286505,9.589943e+06,102.976121,8577641.88,127.870242,89453.590387,93127.829229,67080.829364,-0.039454,0.960546,0.333519


In [113]:
df_veloc_gor_bot = df_veloc1[(df_veloc1["Periodo"]==periodo_analisis)]
df_veloc_gor_bot["Velocidad"] = df_veloc_gor_bot["venta_unidad"] / df_veloc_gor_bot["tiempo_escaneo"]*60
df_veloc_gor_bot["VelocidadMeta"] = df_veloc_gor_bot["Objetivo"]
df_veloc_gor_bot["VelocidadAA"] = df_veloc_gor_bot["venta_unidad_aa"] / df_veloc_gor_bot["tiempo_escaneo_aa"]*60
df_veloc_gor_bot["Cump_Meta"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadMeta"]-1
df_veloc_gor_bot["Cump_AA"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadAA"]-1
df_veloc_gor_bot["Cump_Meta2"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadMeta"]
lista_gor = df_veloc_gor_bot["GOR"].unique().tolist()

df_veloc_gor_bot_top = df_veloc_gor_bot.sort_values(by="Cump_Meta", ascending=False)
df_veloc_gor_bot_bot = df_veloc_gor_bot.sort_values(by="Cump_Meta", ascending=True)


In [114]:
df_veloc_gor_bot_top.head(2)

,regional,supervisor,Date - Año,Date - Mes,nombre,Unid x Minuto,Meta Promedio,venta_unidad,tiempo_escaneo,Objetivo,GOS,GOR,mes,año,Periodo,Tienda,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_AA,Cump_Meta2
2376,Sofía Villanueva,MELANY ALVARADO,2026.0,julio,Paita - PVH,15.079735,14.0,205467.67,817525.0,14.0,Sofía V,Melany A,07,2026,202607,Paita,503975.05,2113233.0,15.079735,14.0,14.309119,0.077124,0.053855,1.077124
2371,Sofía Villanueva,MELANY ALVARADO,2026.0,julio,Chiclayo Aventura - PVH,14.569405,14.0,182185.79,750281.0,14.0,Sofía V,Melany A,07,2026,202607,Chiclayo Aventura,433817.29,1727050.0,14.569405,14.0,15.071386,0.040672,-0.033307,1.040672


In [115]:
graff.Barras_GOR(
    df_veloc_gor_bot_top.head(6),
    gor="nombre",
    ancho="Cump_Meta2",
    cump_meta="Cump_Meta",
    cump_aa="Cump_AA",
    prod="Velocidad",
    titulo="Velocidad por GOR",
    archivo_html=r"plenario/Barras_top_velocidad.html"
)

Archivo generado: plenario/Barras_top_velocidad.html


In [116]:
graff.Barras_GOR(
    df_veloc_gor_bot_bot.head(6),
    gor="nombre",
    ancho="Cump_Meta2",
    cump_meta="Cump_Meta",
    cump_aa="Cump_AA",
    prod="Velocidad",
    titulo="Velocidad por GOR",
    archivo_html=r"plenario/Barras_bot_velocidad.html"
)

Archivo generado: plenario/Barras_bot_velocidad.html


In [117]:
print("finish")

finish
